In [4]:
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from scipy.sparse import hstack

DATASET_PATH = "../datasets"

df = pd.read_csv(f"{DATASET_PATH}/labeled_data/final_training_dataset_ready.csv")


# Fixed Epoch: 01/01/2026
REF_DATE = 1767225600 
ONE_YEAR = 365 * 86400

df["recency_score"] = 1 / ((REF_DATE - df["mod_time_unix"]) / ONE_YEAR + 1)
df["size_logged"] = np.log1p(df["size_bytes"])
df["name_len"] = df["filename"].apply(len)
df["path_len"] = df["path"].apply(len)
df["path_depth"] = df["path"].apply(lambda p: p.count("/") + p.count("\\"))

# Initialize HashingVectorizer
hashing_vectorizer = HashingVectorizer(
    n_features=2048,
    analyzer="char",
    norm=None,
    alternate_sign=False,
    ngram_range=(4, 4),
)

# Apply feature hashing to the 'filename' column
filename_vectors = hashing_vectorizer.fit_transform(df["filename"])
path_vectors = hashing_vectorizer.fit_transform(df["path"])

df

,filename,path,extension,size_bytes,mod_time_unix,source_user,source_type,label_finance,label_hr,label_it,label_general,recency_score,size_logged,name_len,path_len,path_depth
0,MDPI-WINDEX Rpt.docx,C:\Users\omiller\Desktop\Writing\Windex\Docx,.docx,576716,1.744225e+09,omiller,govdoc_real,0,0,0,0,0.578252,13.265107,20,44,6
1,README.md,\\wsl$\Debian\home\torreselaine\git\CSS.NLogEx...,.md,217,1.751013e+09,torreselaine,github_real,0,0,0,0,0.660455,5.384495,9,54,7
2,identity,C:\Users\martin05\Desktop\Coding\puroextremo.c...,NaN,64074,1.755964e+09,dev_injector,secret_injection,0,0,1,1,0.736856,11.067810,8,74,7
3,settings.py,C:\Work\Projects\Mixly_Arduino\mixly_arduino\b...,.py,1773,1.739566e+09,structural_toxic_gen,dependency_noise,0,0,0,0,0.532742,7.480992,11,102,11
4,17acad98.lib,C:\Work\Projects\opensplice\src\api\dcps\java\...,.lib,103820,1.735757e+09,structural_toxic_gen,dependency_noise,0,0,0,0,0.500537,11.550424,12,89,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5144,docker-compose.yml,C:\Users\wilsonspencer\Desktop\Coding\carbon-a...,.yml,47672,1.680493e+09,structural_toxic_gen,dependency_noise,0,0,0,0,0.266648,10.772120,18,187,22
5145,cathycvr.docx,Z:\HR\Recruiting\Reqs\2023\Docx\Cathycvr,.docx,5179965,1.645951e+09,ybrown,govdoc_real,0,0,0,0,0.206374,15.460309,13,40,6
5146,ACKDisabledSplitViewDelegate.h,C:\Work\Projects\AndromedaIDE\AndromedaCodeKit,.h,1449,1.747482e+09,devops,github_real,0,0,0,0,0.614977,7.279319,30,46,4
5147,QuickTimeVRLocalized.qtr,C:\Program Files\QuickTime\QTSystem\QuickTimeV...,.qtr,29696,1.749900e+09,jo,system_noise,0,0,0,0,0.645421,10.298801,24,69,5


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
from scipy.sparse import hstack

# Define target-specific valuable extensions
target_extension_map = {
    "label_finance": [".xlsx", ".xls", ".csv", ".pdf", ".accp", ".pptx"],
    "label_hr": [".docx", ".doc", ".rtf", ".pdf", ".xlsx", ".xls"],
    "label_it": [".pem",".key",".kdbx",".p12",".ovpn",".private",".wallet",".sql",".env"]
}

# Add "label_general" after defining the other labels
target_extension_map["label_general"] = list(
    set(
        target_extension_map["label_finance"]
        + target_extension_map["label_hr"]
        + target_extension_map["label_it"]
    )
)
common_dev_junk = [".pyc", ".pyi", ".lcl", ".pyd", ".map", ".ts", ".js", ".spec", ".test.js",]

# Define target-specific junk extensions
target_junk_map = {
    "label_finance": [".class", ".java", ".cpp", ".obj", ".dll", ".exe"]
    + common_dev_junk,
    "label_hr": [".class", ".java", ".py", ".obj", ".dll", ".exe"] + common_dev_junk,
    "label_it": [".tmp", ".log", ".cache"]
    + [ext for ext in common_dev_junk if ext not in [".ts", ".js"]],
}

target_junk_map["label_general"] = list(
    set(
        target_junk_map["label_finance"]
        + target_junk_map["label_hr"]
        + target_junk_map["label_it"]
    )
)

c_grid = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
weight_grid = [1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0, 15.0, 18.0]

for target in ["label_finance", "label_hr", "label_it", "label_general"]:
    print(f"\n" + "="*50)
    print(f"📊 GRID SEARCH RESULTS FOR: {target}")
    print("="*50)

    # 1. Prepare Data
    df_temp = df.copy()
    df_temp["valuable_ext"] = df_temp["extension"].isin(target_extension_map[target]).astype(int)
    df_temp["junk_ext"] = df_temp["extension"].isin(target_junk_map[target]).astype(int)

    numerical_features = df_temp[
        ["recency_score", "size_logged", "valuable_ext", "junk_ext", 
         "name_len", "path_len", "path_depth"]
    ].values
    
    X_target = hstack([numerical_features, filename_vectors, path_vectors])
    y_target = df_temp[target].values

    # 2. Split Data
    X_train, X_test, y_train, y_test = train_test_split(
        X_target, y_target, test_size=0.2, random_state=42, stratify=y_target
    )

    all_results = []

    # 3. Iterative Testing
    for C in c_grid:
        for W in weight_grid:
            model = LogisticRegression(solver="lbfgs", max_iter=1000, C=C, class_weight={0: 1, 1: W})
            model.fit(X_train, y_train)
            
            y_pred = model.predict(X_test)
            p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
            
            # Record every run to see the trends
            all_results.append({
                "C": C,
                "Weight": W,
                "Precision": round(p, 3),
                "Recall": round(r, 3),
                "F1": round(f1, 3),
                "Bias": round(model.intercept_[0], 2)
            })

            print(f"Tested C={C}, Weight={W} => P={p:.3f}, R={r:.3f}, F1={f1:.3f}")

    # 4. Create Table and Filter
    results_df = pd.DataFrame(all_results)
    
    # Filter for candidates that are close to your goals
    # We use a slightly relaxed filter so you can see 'near misses'
    candidates = results_df[(results_df['Precision'] >= 0.60) & (results_df['Recall'] >= 0.75)]
    
    if not candidates.empty:
        # Sort by Recall then Precision to prioritize discovery
        sorted_candidates = candidates.sort_values(by=['Recall', 'Precision'], ascending=False)
        print(sorted_candidates.to_string(index=False))
    else:
        print("⚠️ No configurations met the baseline (P>=0.6, R>=0.75).")
        print("Top 5 by F1 Score instead:")
        print(results_df.sort_values(by='F1', ascending=False).head(5).to_string(index=False))


📊 GRID SEARCH RESULTS FOR: label_finance
Tested C=0.01, Weight=1.0 => P=0.583, R=0.194, F1=0.292
Tested C=0.01, Weight=1.5 => P=0.613, R=0.528, F1=0.567
Tested C=0.01, Weight=2.0 => P=0.605, R=0.639, F1=0.622
Tested C=0.01, Weight=3.0 => P=0.643, R=0.750, F1=0.692
Tested C=0.01, Weight=4.0 => P=0.659, R=0.806, F1=0.725
Tested C=0.01, Weight=6.0 => P=0.681, R=0.889, F1=0.771
Tested C=0.01, Weight=8.0 => P=0.667, R=0.944, F1=0.782
Tested C=0.01, Weight=12.0 => P=0.642, R=0.944, F1=0.764
Tested C=0.01, Weight=15.0 => P=0.630, R=0.944, F1=0.756
Tested C=0.01, Weight=18.0 => P=0.630, R=0.944, F1=0.756
Tested C=0.05, Weight=1.0 => P=0.677, R=0.583, F1=0.627
Tested C=0.05, Weight=1.5 => P=0.667, R=0.722, F1=0.693
Tested C=0.05, Weight=2.0 => P=0.650, R=0.722, F1=0.684
Tested C=0.05, Weight=3.0 => P=0.674, R=0.861, F1=0.756
Tested C=0.05, Weight=4.0 => P=0.681, R=0.889, F1=0.771
Tested C=0.05, Weight=6.0 => P=0.660, R=0.917, F1=0.767
Tested C=0.05, Weight=8.0 => P=0.667, R=0.944, F1=0.782
Tes

In [ ]:
print("\ntarget_configs = {")
for key, value in final_configs.items():
    print(f'    "{key}": {{"C": {value["C"]}, "weight": {value["weight"]}}},')
print("}")


target_configs = {
    "label_finance": {"C": 0.8, "weight": 15.0},
    "label_hr": {"C": 1.0, "weight": 1.5},
    "label_it": {"C": 1.0, "weight": 12.0},
    "label_general": {"C": 0.2, "weight": 2.0},
}
